In [ ]:
from datasets import load_dataset

ds = load_dataset("codeparrot/codeparrot-clean", split="train", streaming=True)
print(ds)
code_ds = ds.take(20000)
list_code_ds = list(code_ds["content"])

print(len(list_code_ds))
print(list_code_ds[0])

In [ ]:
list_code_ds = [str(item) for item in list_code_ds if item is not None]


In [ ]:
#training tokenizer using sewntence piece
import sentencepiece as spm

sp = spm.SentencePieceTrainer.train(
    model_prefix= "code_fim_tokenizer",
    sentence_iterator = iter(list_code_ds),
    vocab_size=32000,
    character_coverage=1.0,
    max_sentence_length=150000,
    pad_id =0,
    bos_id=-1,  
    eos_id=-1, 
    unk_id=1,
    user_defined_symbols=['<PRE>', '<SUF>', '<MID>', '<EOT>'] 
)


In [ ]:
sp = spm.SentencePieceProcessor(model_file='/kaggle/working/code_fim_tokenizer.model')
print(sp.get_piece_size())  # should be 32000
print(sp.piece_to_id('<PRE>'))   # should not be 0 or 1
print(sp.piece_to_id('<SUF>'))
print(sp.piece_to_id('<MID>'))
print(sp.piece_to_id('<EOT>'))
print(sp.encode('<PRE>', out_type=str))  # should be ['<PRE>'] not split
print(sp.encode('<MID>', out_type=str))  # should be ['<MID>']

example = "def add(a, b):\n    return a + b"
print(sp.encode(example, out_type=str))
print(sp.decode(sp.encode(example)))  # should reconstruct original
fim = "<PRE>def add(a, b):\n    return <SUF>\n<MID>a + b<EOT>"
tokens = sp.encode(fim, out_type=str)
print(tokens)  # PRE, SUF, MID, EOT should each appear as single tokens


In [ ]:
import torch as pt
import random
from datasets import load_dataset
import sentencepiece as spm


random.seed(42)

PRE_ID = sp.piece_to_id('<PRE>')
SUF_ID = sp.piece_to_id('<SUF>')
MID_ID = sp.piece_to_id('<MID>')
EOT_ID = sp.piece_to_id('<EOT>')

def FimTransform(chunk, chunk_list):
    if random.random() > 0.5:
        ar_chunk = chunk[0:len(chunk)-1]
        ar_chunk.append(EOT_ID)
        chunk_list.append(ar_chunk)
    else:
        fim_chunk = chunk[0:len(chunk)-4]
        r1 = random.randint(0, len(fim_chunk))
        r2 = random.randint(0, len(fim_chunk))
        pos1 = min(r1, r2)
        pos2 = max(r1, r2)
        prefix = [PRE_ID] + fim_chunk[0:pos1]
        suffix = [SUF_ID] + fim_chunk[pos2:len(fim_chunk)]
        middle = [MID_ID] + fim_chunk[pos1:pos2] + [EOT_ID]
        chunk_list.append(prefix + suffix + middle)
    return chunk_list

ds = load_dataset("codeparrot/codeparrot-clean", split="train", streaming=True)

chunk_list = []
file_counter = 0
save_counter = 0

for row in ds.take(900000):
    file_counter += 1
    data = row["content"]
    encoded_data = sp.encode(data)

    if len(encoded_data) > 512:
        for i in range(0, len(encoded_data), 512):
            batch = encoded_data[i:i+512]
            if len(batch) == 512:
                chunk_list = FimTransform(batch, chunk_list)

    if file_counter % 50000 == 0:
        save_counter += 1
        pt.save(chunk_list, f'chunks_{save_counter}.pt')
        print(f'Saved chunks_{save_counter}.pt | files: {file_counter} | chunks: {len(chunk_list)}')
        chunk_list = []

if chunk_list:
    save_counter += 1
    pt.save(chunk_list, f'chunks_{save_counter}.pt')
    print(f'Saved final chunks_{save_counter}.pt | files: {file_counter} | chunks: {len(chunk_list)}')

print(f'Done. Total files: {file_counter}, Total saves: {save_counter}')

In [ ]:
chunks = pt.load('chunks_1.pt', weights_only=False)
print(f'Total chunks: {len(chunks)}')
print(f'Chunk 0 length: {len(chunks[0])}')
print(f'Chunk 0 first 5 tokens: {chunks[0][:5]}')
print(f'Chunk 0 last 5 tokens: {chunks[0][-5:]}')

# Check a few for sentinel tokens
PRE_ID = sp.piece_to_id('<PRE>')
EOT_ID = sp.piece_to_id('<EOT>')

fim_count = sum(1 for c in chunks[:1000] if c[0] == PRE_ID)
eot_count = sum(1 for c in chunks[:1000] if EOT_ID in c)

print(f'FIM examples in first 1000: {fim_count} ({fim_count/10:.1f}%)')
print(f'EOT present in first 1000: {eot_count} ({eot_count/10:.1f}%)')